## 01: Skrip Ingesti Data (Data Ingestion) - Versi Cloud

Tujuan notebook ini adalah untuk memuat "Knowledge Base" kita ke **ChromaDB Cloud**.

**Proses:**
1.  **Muat Kredensial**: Baca `.env` dari folder `backend/` untuk mendapatkan API Key, Tenant, dan Database.
2.  **Siapkan Data**: Kita akan menggunakan data teks sederhana (dummy).
3.  **Siapkan Embeddings**: Kita akan memuat model embedding lokal (`all-MiniLM-L6-v2`).
4.  **Koneksi & Ingest**: Kita akan terhubung ke CloudClient dan menggunakan LangChain untuk memuat dokumen.

In [ ]:
import os
import chromadb
from dotenv import load_dotenv
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

print("Library berhasil diimpor.")

Library berhasil diimpor.


### Langkah 1: Muat Kredensial & Konfigurasi

In [2]:
# --- KODE BARU (YANG BENAR) ---

# Path ke file .env.
# Logika ini berarti: "Dari folder 'notebooks/' saat ini,
# naik satu level ('..') ke folder root 'RAG',
# lalu masuk ke folder 'backend', dan temukan file '.env'".
dotenv_path = os.path.join('..', 'backend', '.env')

# Periksa apakah file .env ada sebelum mencoba memuatnya
if not os.path.exists(dotenv_path):
    raise FileNotFoundError(f"File .env tidak ditemukan di path yang diharapkan: {os.path.abspath(dotenv_path)}")

# Muat file .env
load_dotenv(dotenv_path=dotenv_path)

# Baca variabel yang kita butuhkan
CHROMA_API_KEY = os.getenv('CHROMA_API_KEY')
CHROMA_TENANT = os.getenv('CHROMA_TENANT')
CHROMA_DATABASE = os.getenv('CHROMA_DATABASE')
EMBEDDING_MODEL_NAME = os.getenv('EMBEDDING_MODEL', 'all-MiniLM-L6-v2')

COLLECTION_NAME = "chart_knowledge" # Nama 'folder' di dalam database Anda

if not CHROMA_API_KEY:
    raise ValueError("CHROMA_API_KEY ditemukan sebagai None. Pastikan nilainya ada di dalam file .env.")

print("Variabel .env berhasil dimuat.")
print(f"Tenant: {CHROMA_TENANT}, Database: {CHROMA_DATABASE}")

Variabel .env berhasil dimuat.
Tenant: 6d4f29ca-1eef-4130-bfb1-1af19306b943, Database: RAG_rope


### Langkah 2 & 3: Siapkan Data & Model Embedding

In [6]:
# Data Dummy (Contoh Knowledge Base kita)
DUMMY_DATA = [
    {
        "text": "Head and Shoulders adalah pola pembalikan arah (reversal pattern) bearish. Ini terdiri dari tiga puncak, dengan puncak tengah (head) lebih tinggi dari dua puncak lainnya (shoulders). Garis leher (neckline) ditarik menghubungkan titik terendah dari dua lembah. Jika harga menembus di bawah neckline, itu adalah sinyal jual.",
        "source": "Buku Analisis Teknis, Hal. 45"
    },
    {
        "text": "Bullish Flag adalah pola kelanjutan (continuation pattern). Ini muncul setelah pergerakan naik yang kuat (tiang bendera), diikuti oleh konsolidasi miring ke bawah (bendera). Pola ini menandakan bahwa tren naik kemungkinan akan berlanjut setelah harga menembus ke atas garis resisten bendera.",
        "source": "Investopedia - Pola Chart"
    }
]

# Ubah data dummy menjadi format Dokumen LangChain
documents = []
for item in DUMMY_DATA:
    doc = Document(page_content=item["text"], metadata={"source": item["source"]})
    documents.append(doc)

print(f"Siap untuk memproses {len(documents)} dokumen.")

# Inisialisasi model embedding (Sama seperti sebelumnya)
print("Memuat model embedding... (Mungkin butuh waktu saat pertama kali)")
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)
print("Model embedding berhasil dimuat.")

Siap untuk memproses 2 dokumen.
Memuat model embedding... (Mungkin butuh waktu saat pertama kali)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

C:\Users\roofi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model embedding berhasil dimuat.


### Langkah 4: Koneksi & Ingest ke ChromaDB Cloud

Alih-alih `persist_directory`, kita sekarang akan menggunakan `CloudClient` dan LangChain adapter.

In [7]:
print("Menghubungkan ke ChromaDB Cloud...")

# 1. Inisialisasi Klien Cloud
client = chromadb.CloudClient(
    api_key=CHROMA_API_KEY,
    tenant=CHROMA_TENANT,
    database=CHROMA_DATABASE
)

print(f"Terhubung ke Database: {CHROMA_DATABASE}")

# 2. Gunakan LangChain adapter untuk memuat dokumen
# Ini adalah cara termudah. LangChain akan menangani koneksi, embedding, dan penambahan data.
print(f"Menyimpan dokumen ke collection: {COLLECTION_NAME}...")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    client=client, # <-- Berikan Klien Cloud
    collection_name=COLLECTION_NAME # <-- Tentukan nama collection
)

print("--- SELESEI! ---")
print(f"Berhasil menyimpan {len(documents)} dokumen ke ChromaDB Cloud.")

Menghubungkan ke ChromaDB Cloud...
Terhubung ke Database: RAG_rope
Menyimpan dokumen ke collection: chart_knowledge...
--- SELESEI! ---
Berhasil menyimpan 2 dokumen ke ChromaDB Cloud.


### Verifikasi (Opsional)

Mari kita coba lakukan pencarian sederhana.

In [8]:
print("Melakukan tes pencarian...")

# 'vectorstore' yang kita buat di atas sudah siap untuk di-query
query = "Apa itu pola bendera?"
results = vectorstore.similarity_search(query, k=1)

if results:
    print(f"Query: {query}")
    print(f"Hasil teratas: {results[0].page_content}")
    print(f"Sumber: {results[0].metadata.get('source')}")
else:
    print("Tes pencarian gagal.")

Melakukan tes pencarian...
Query: Apa itu pola bendera?
Hasil teratas: Bullish Flag adalah pola kelanjutan (continuation pattern). Ini muncul setelah pergerakan naik yang kuat (tiang bendera), diikuti oleh konsolidasi miring ke bawah (bendera). Pola ini menandakan bahwa tren naik kemungkinan akan berlanjut setelah harga menembus ke atas garis resisten bendera.
Sumber: Investopedia - Pola Chart
